# From Exposure to Action
### A Data-Driven Career Compass for the AI Era

**Will AI replace human jobs, and which occupations are most at risk of AI-driven displacement, transformation, or augmentation — and what should students and working professionals do about it?**

This notebook is the Phase 1 analysis deliverable. See `docs/From_Exposure_to_Action_Data_Dictionary.xlsx` for the full research framework (typology, hypotheses, target variables, guardrails) and `README.md` for the project overview.

**Status: data acquisition (AIOE, O*NET) and the AIOE+O*NET master join are implemented and runnable end to end. Remaining sources (ILO, OECD, BLS OEWS, GPTs are GPTs, Anthropic Economic Index, Statistics Canada) are still TODO — see `src/data_acquisition/README.md`. RQ sections below will fill in as each source is added.**

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Data acquisition

Run once (or re-run to refresh) before this notebook can execute end to end:

```bash
python ../src/data_acquisition/fetch_aioe.py
python ../src/data_acquisition/fetch_onet.py
python ../src/data_acquisition/fetch_ilo_genai.py
python ../src/data_acquisition/fetch_bls_crosswalks.py
```

(Remaining fetchers -- OECD, BLS OEWS, GPTs are GPTs, Anthropic Economic Index, Statistics Canada -- are tracked in `src/data_acquisition/README.md`.)

## 2. Build the master occupation-level dataset

Joins AIOE + O*NET + ILO on occupation code via `src/crosswalks/` and `src/analysis/build_master_dataset.py`. Produces the Composite Exposure Score and a first-pass (provisional) Impact Pattern classification -- see that script's module docstring for the classification rule and its caveats.

In [ ]:
import sys
sys.path.insert(0, "../src")
from analysis.build_master_dataset import merge_all, save

# Run once (or re-run after adding a new source) to rebuild data/processed/master_occupations.csv.
# Requires fetch_aioe.py and fetch_onet.py to have been run first (see Section 1).
master = merge_all()
save(master)
master.head()

## 3. RQ1 — Consolidated risk ranking across indices

Top/bottom occupations by composite AI exposure. **Currently uses AIOE + ILO only** (the two indices joined so far in `build_master_dataset.py`) -- OECD and GPTs-are-GPTs are tracked as `[ ] later pass` in `src/data_acquisition/README.md` and will extend the correlation matrix once added. Tests H1 with the sources on hand; not the full picture yet.

In [ ]:
master = pd.read_csv("../data/processed/master_occupations.csv")

N = 15
CATEGORY_COLORS = {
    "Automation": "#2a78d6",       # blue
    "Transformation": "#eb6834",   # orange
    "Augmentation": "#1baf7a",     # aqua
}
MISSING_COLOR = "#9a9993"  # occupations with no impact_pattern yet

ranked = master.dropna(subset=["composite_exposure_score"]).sort_values(
    "composite_exposure_score", ascending=False
)
print(f"{len(ranked)} of {len(master)} occupations have a composite exposure score.")

display_cols = ["soc_code", "occupation_title", "composite_exposure_score",
                "aioe_score", "ilo_score", "impact_pattern"]
top_n = ranked.head(N)[display_cols]
bottom_n = ranked.tail(N)[display_cols]

print(f"\nTop {N} highest-exposure occupations:")
display(top_n)
print(f"\nBottom {N} lowest-exposure occupations:")
display(bottom_n)

In [ ]:
# Do the two available indices agree on which occupations are exposed?
# AIOE (Felten/Raj/Seamans, LLM-linguistic-task-based) and ILO (Gmyrek et al.,
# GPT-4o/Gemini task-rating-based) use different methodologies, so agreement here
# is informative even with just these two -- a low/insignificant rho would say the
# indices disagree more than expected, which matters as much as a high one.
paired = master.dropna(subset=["aioe_score", "ilo_score"])
rho, p_value = stats.spearmanr(paired["aioe_score"], paired["ilo_score"])
print(f"Spearman rho (AIOE vs ILO), n={len(paired)}: {rho:.3f} (p={p_value:.2e})")

print("\nimpact_pattern distribution (occupations with a value):")
print(master["impact_pattern"].value_counts())
print(f"({master['impact_pattern'].isna().sum()} occupations have no impact_pattern -- "
      "no ILO task-level match, see build_master_dataset.py)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_df = top_n.iloc[::-1]  # reverse so the highest-exposure occupation plots on top
colors = plot_df["impact_pattern"].map(CATEGORY_COLORS).fillna(MISSING_COLOR)
ax.barh(plot_df["occupation_title"], plot_df["composite_exposure_score"], color=colors)
ax.set_xlabel("Composite exposure score (z-scored mean of AIOE + ILO)")
ax.set_title(f"Top {N} occupations by AI exposure")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CATEGORY_COLORS.values()]
ax.legend(handles, CATEGORY_COLORS.keys(), title="Impact pattern", loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for pattern, color in CATEGORY_COLORS.items():
    subset = paired[paired["impact_pattern"] == pattern]
    ax.scatter(subset["aioe_score"], subset["ilo_score"], s=20, alpha=0.6,
               color=color, label=pattern)
no_pattern = paired[paired["impact_pattern"].isna()]
if len(no_pattern):
    ax.scatter(no_pattern["aioe_score"], no_pattern["ilo_score"], s=20, alpha=0.4,
               color=MISSING_COLOR, label="(no impact_pattern)")
ax.set_xlabel("AIOE score")
ax.set_ylabel("ILO GenAI exposure score")
ax.set_title(f"AIOE vs. ILO exposure (Spearman ρ={rho:.2f}, n={len(paired)})")
ax.legend(title="Impact pattern")
plt.tight_layout()
plt.show()

**Reading this section:** the ranking above is associative, not predictive -- a high composite score describes an occupation's task content as measured by two LLM-exposure methodologies today, not a forecast of job loss (see the project's guardrails in `README.md`). The Spearman correlation is the key sanity check: if AIOE and ILO disagreed strongly (rho near 0 or negative) it would say the two methodologies are capturing different things and the composite score should be read cautiously until OECD and GPTs-are-GPTs are added to triangulate further. Revisit both the ranking and this correlation once those sources join the master table.

**A caveat on ties in the ranking:** the ISCO-08 -> SOC 2018 crosswalk is many-to-many, so several distinct SOC titles here can trace back to the very same ISCO-08 unit group and inherit its exact `ilo_score`. When one of those tied occupations also has no AIOE coverage (`aioe_score` is NaN), its `composite_exposure_score` is driven entirely by that one shared ILO value -- so an exact tie in the top/bottom list reflects a shared ISCO parent, not two independent methodologies agreeing on that specific occupation. Worth noting explicitly in the write-up rather than presenting every entry as independently confirmed.

## 4. RQ2 -- Automation vs. Transformation vs. Augmentation

`impact_pattern` in the master table is now populated from ILO's task-level score variance (see `build_master_dataset.py`'s module docstring for the classification rule). This is a first-pass, provisional heuristic -- cross-check it here against the Anthropic Economic Index's own automation/augmentation usage split once that source is added (tests H2).

## 5. RQ3 — Task-level vulnerability

Which tasks within an occupation carry the highest ILO exposure scores.

## 6. RQ4 — Protective skill profile

Compare O*NET skill/ability distributions between high- and low-exposure occupation groups (tests H4).

## 7. RQ5 — Theoretical exposure vs. realized labor-market impact

Correlate composite exposure score against BLS OEWS multi-year employment/wage growth, and against Anthropic Economic Index actual-usage share (tests H3). Associative only — not a forecast.

## 8. Student layer — education path vs. exposure

Join O*NET Job Zones (education/experience/training required) to the composite exposure score; aggregate by education level.

## 9. Professional layer — reskilling paths

Occupation-to-occupation skill-vector similarity to surface, for a given high-exposure occupation, the nearest lower-exposure alternatives and the skill gap to close.

## 10. Appendix — Canada in context

One comparison chart: Statistics Canada's own NOC-based AI exposure estimates against the equivalent global/US-ranked occupations. Not a parallel pipeline (see README guardrails).

## 11. Summary & what this means for you

Decision-support framing only — evidence, risks, opportunities, skills, and alternatives; never a single prescriptive recommendation. Sets up the Phase 2 web app's dashboard fields (see the Data Dictionary's "Phase 2 Tool Design" tab).